# Notebook 1: Data Collection für Photonic Benchmarks

## Ziel
Öffentlich verfügbare experimentelle Daten aus Papers sammeln, herunterladen und in eine DuckDB-Datenbank einpflegen.  
Dies dient als Ground-Truth / Vergleichsbasis für synthetische Daten.

In [1]:
# Zelle 1: Imports
import pandas as pd
import requests
import duckdb
import json
from pathlib import Path
import time
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotheken geladen")

✅ Bibliotheken geladen


In [4]:
# Zelle 2: Konfiguration & bekannte öffentliche Datenquellen

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect("photonic_benchmark.db")

# Tabelle für Rohdaten anlegen
con.execute("""
CREATE TABLE IF NOT EXISTS raw_papers (
    paper_id VARCHAR,
    title VARCHAR,
    year INTEGER,
    source VARCHAR,
    doi VARCHAR,
    matrix_size INTEGER,
    mae FLOAT,
    snr_db FLOAT,
    effective_bits FLOAT,
    noise_level FLOAT,
    comments TEXT,
    data_url VARCHAR,
    retrieved_at TIMESTAMP
)
""")

print("✅ Datenbank-Tabelle bereit")

✅ Datenbank-Tabelle bereit


In [5]:
# Zelle 3: Manuelle Sammlung wichtiger Papers (mit Links zu Supplementary Data)

papers = [
    {
        "paper_id": "cambridge_pip_2024",
        "title": "I/O-efficient iterative matrix inversion with photonic integrated circuits",
        "year": 2024,
        "source": "Nature Communications",
        "doi": "10.1038/s41467-024-50302-3",
        "data_url": "https://doi.org/10.17863/CAM.109538",
        "comments": "Enthält Supplementary Data mit Matrix-Inversion Benchmarks",
        "matrix_size": 64,
        "mae": None,  # manuell später eintragen
        "snr_db": None
    },
    {
        "paper_id": "lightmatter_2025",
        "title": "Photonic AI Accelerator Benchmarks (Lightmatter)",
        "year": 2025,
        "source": "arXiv / Company Paper",
        "doi": None,
        "data_url": "https://lightmatter.co/research",  # oft Code + Results
        "comments": "ResNet / BERT Benchmarks, Precision & Energy",
        "matrix_size": 512,
        "mae": None
    },
    {
        "paper_id": "mnist_photonic_2023",
        "title": "Photonic Neural Network MNIST Classification",
        "year": 2023,
        "source": "Various (z.B. Peserico et al.)",
        "doi": None,
        "data_url": None,
        "comments": "Accuracy ~94% auf MNIST mit photonic Hardware",
        "matrix_size": 28*28,
        "mae": 0.05  # Beispielwert aus Papers
    }
]

# In Datenbank einfügen
for p in papers:
    con.execute("""
        INSERT INTO raw_papers 
        (paper_id, title, year, source, doi, matrix_size, mae, snr_db, comments, data_url, retrieved_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
    """, [p["paper_id"], p["title"], p["year"], p["source"], p["doi"], 
          p.get("matrix_size"), p.get("mae"), p.get("snr_db"), 
          p["comments"], p.get("data_url")])

print("✅ Bekannte Papers in DB eingetragen")
con.execute("SELECT * FROM raw_papers").df()

✅ Bekannte Papers in DB eingetragen


,paper_id,title,year,source,doi,matrix_size,mae,snr_db,effective_bits,noise_level,comments,data_url,retrieved_at
0,cambridge_pip_2024,I/O-efficient iterative matrix inversion with ...,2024,Nature Communications,10.1038/s41467-024-50302-3,64,NaN,NaN,NaN,NaN,Enthält Supplementary Data mit Matrix-Inversio...,https://doi.org/10.17863/CAM.109538,2026-07-04 09:14:18.904290
1,lightmatter_2025,Photonic AI Accelerator Benchmarks (Lightmatter),2025,arXiv / Company Paper,None,512,NaN,NaN,NaN,NaN,"ResNet / BERT Benchmarks, Precision & Energy",https://lightmatter.co/research,2026-07-04 09:14:18.917258
2,mnist_photonic_2023,Photonic Neural Network MNIST Classification,2023,Various (z.B. Peserico et al.),None,784,0.05,NaN,NaN,NaN,Accuracy ~94% auf MNIST mit photonic Hardware,None,2026-07-04 09:14:18.922418
3,cambridge_pip_2024,I/O-efficient iterative matrix inversion with ...,2024,Nature Communications,10.1038/s41467-024-50302-3,64,NaN,NaN,NaN,NaN,Enthält Supplementary Data mit Matrix-Inversio...,https://doi.org/10.17863/CAM.109538,2026-07-04 09:41:02.175279
4,lightmatter_2025,Photonic AI Accelerator Benchmarks (Lightmatter),2025,arXiv / Company Paper,None,512,NaN,NaN,NaN,NaN,"ResNet / BERT Benchmarks, Precision & Energy",https://lightmatter.co/research,2026-07-04 09:41:02.184007
5,mnist_photonic_2023,Photonic Neural Network MNIST Classification,2023,Various (z.B. Peserico et al.),None,784,0.05,NaN,NaN,NaN,Accuracy ~94% auf MNIST mit photonic Hardware,None,2026-07-04 09:41:02.191625
6,cambridge_pip_2024,I/O-efficient iterative matrix inversion with ...,2024,Nature Communications,10.1038/s41467-024-50302-3,64,NaN,NaN,NaN,NaN,Enthält Supplementary Data mit Matrix-Inversio...,https://doi.org/10.17863/CAM.109538,2026-07-04 09:41:16.935210
7,lightmatter_2025,Photonic AI Accelerator Benchmarks (Lightmatter),2025,arXiv / Company Paper,None,512,NaN,NaN,NaN,NaN,"ResNet / BERT Benchmarks, Precision & Energy",https://lightmatter.co/research,2026-07-04 09:41:16.941308
8,mnist_photonic_2023,Photonic Neural Network MNIST Classification,2023,Various (z.B. Peserico et al.),None,784,0.05,NaN,NaN,NaN,Accuracy ~94% auf MNIST mit photonic Hardware,None,2026-07-04 09:41:16.946300
9,cambridge_pip_2024,I/O-efficient iterative matrix inversion with ...,2024,Nature Communications,10.1038/s41467-024-50302-3,64,NaN,NaN,NaN,NaN,Enthält Supplementary Data mit Matrix-Inversio...,https://doi.org/10.17863/CAM.109538,2026-07-04 10:36:30.592458


In [6]:
# Zelle 4: Automatisches Herunterladen von Supplementary Data (Beispiel)

def download_supplementary(doi_or_url, filename):
    try:
        if "doi.org" in doi_or_url:
            resp = requests.get(doi_or_url.replace("doi.org", "data.datacite.org") + "/data", timeout=10)
        else:
            resp = requests.get(doi_or_url, timeout=10)
        
        if resp.status_code == 200:
            path = DATA_DIR / filename
            path.write_bytes(resp.content)
            print(f"✅ Heruntergeladen: {filename}")
            return True
    except Exception as e:
        print(f"⚠️ Fehler bei {filename}: {e}")
    return False

# Beispiel-Aufruf (kann erweitert werden)
# download_supplementary("https://doi.org/10.17863/CAM.109538", "cambridge_pip_supplement.zip")

In [7]:
# Zelle 5: Übersicht & nächste Schritte

print("Aktueller Stand der Rohdaten:")
print(con.execute("SELECT title, year, matrix_size, mae, comments FROM raw_papers").df())

print("\n\nNächste Schritte für dieses Notebook:")
print("1. Manuell weitere Papers hinzufügen (arXiv, Nature, Optica)")
print("2. Supplementary Material herunterladen und parsen")
print("3. In Notebook 2 & 3 mit synthetischen Daten vergleichen")

Aktueller Stand der Rohdaten:
                                                title  year  matrix_size  \
0   I/O-efficient iterative matrix inversion with ...  2024           64   
1    Photonic AI Accelerator Benchmarks (Lightmatter)  2025          512   
2        Photonic Neural Network MNIST Classification  2023          784   
3   I/O-efficient iterative matrix inversion with ...  2024           64   
4    Photonic AI Accelerator Benchmarks (Lightmatter)  2025          512   
5        Photonic Neural Network MNIST Classification  2023          784   
6   I/O-efficient iterative matrix inversion with ...  2024           64   
7    Photonic AI Accelerator Benchmarks (Lightmatter)  2025          512   
8        Photonic Neural Network MNIST Classification  2023          784   
9   I/O-efficient iterative matrix inversion with ...  2024           64   
10   Photonic AI Accelerator Benchmarks (Lightmatter)  2025          512   
11       Photonic Neural Network MNIST Classification  202

In [8]:
con.close()